In [1]:
import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModelForVision2Seq, AutoProcessor
import time
import numpy as np
import cv2
import textwrap
from PIL import Image, ImageDraw, ImageFont
import enum
import json
import os 

/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-22 20:14:44.358295: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-22 20:14:44.387275: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-22 20:14:44.387298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-2

In [2]:
class CotTag(enum.Enum):
    TASK = "TASK:"
    PLAN = "PLAN:"
    VISIBLE_OBJECTS = "VISIBLE OBJECTS:"
    SUBTASK_REASONING = "SUBTASK REASONING:"
    SUBTASK = "SUBTASK:"
    MOVE_REASONING = "MOVE REASONING:"
    MOVE = "MOVE:"
    GRIPPER_POSITION = "GRIPPER POSITION:"
    ACTION = "ACTION:"


def get_cot_tags_list():
    return [
        CotTag.TASK.value,
        CotTag.PLAN.value,
        CotTag.VISIBLE_OBJECTS.value,
        CotTag.SUBTASK_REASONING.value,
        CotTag.SUBTASK.value,
        CotTag.MOVE_REASONING.value,
        CotTag.MOVE.value,
        CotTag.GRIPPER_POSITION.value,
        CotTag.ACTION.value,
    ]



In [3]:
device = "cuda:0"
# Load Processor & VLA
# path_to_converted_ckpt = "Embodied-CoT/ecot-openvla-7b-bridge"
# path_to_converted_ckpt = "Embodied-CoT/ecot-openvla-7b-oxe"
path_to_converted_ckpt = "/home/zhekai/code/embodied-CoT/outputs/ecot-openvla-7b-oxe+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0"
processor = AutoProcessor.from_pretrained(path_to_converted_ckpt, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    path_to_converted_ckpt,
    # attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
    torch_dtype=torch.bfloat16,
    # low_cpu_mem_usage=True,
    trust_remote_code=True,
).to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.49.0` and `tokenizers==0.21.1`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  5.10it/s]


In [4]:
# vla.language_model.save_pretrained("logs/llama-bridge")
from vllm import LLM, SamplingParams
vla.input_embds = vla.language_model.get_input_embeddings()


# load language model with VLLM
if hasattr(vla, "language_model"):
    del vla.language_model
vla.language_model = LLM("../logs/_home_zhekai_code_embodied-CoT_outputs_ecot-openvla-7b-oxe+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0-vllm",  
                         trust_remote_code=True, 
                         gpu_memory_utilization=0.7, 
                         enable_chunked_prefill = True, 
                         preemption_mode='swap', swap_space = 10
                         )

INFO 03-22 20:14:57 __init__.py:186] Automatically detected platform cuda.


2025-03-22 20:14:57,784	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 03-22 20:15:03 config.py:542] This model supports multiple tasks: {'embed', 'score', 'reward', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 03-22 20:15:03 config.py:1557] Chunked prefill is enabled with max_num_batched_tokens=5120.
INFO 03-22 20:15:03 llm_engine.py:234] Initializing a V0 LLM engine (v0.1.dev4429+ga257914) with config: model='../logs/_home_zhekai_code_embodied-CoT_outputs_ecot-openvla-7b-oxe+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0-vllm', speculative_config=None, tokenizer='../logs/_home_zhekai_code_embodied-CoT_outputs_ecot-openvla-7b-oxe+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0-vllm', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eag

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:00,  2.26it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.71it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.63it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.69it/s]



INFO 03-22 20:15:06 model_runner.py:1119] Loading model weights took 12.5528 GB
INFO 03-22 20:15:07 worker.py:267] Memory profiling takes 0.87 seconds
INFO 03-22 20:15:07 worker.py:267] the current vLLM instance can use total_gpu_memory (23.64GiB) x gpu_memory_utilization (0.70) = 16.55GiB
INFO 03-22 20:15:07 worker.py:267] model weights take 12.55GiB; non_torch_memory takes 0.08GiB; PyTorch activation peak memory takes 0.44GiB; the rest of the memory reserved for KV Cache is 3.48GiB.
INFO 03-22 20:15:07 executor_base.py:110] # CUDA blocks: 445, # CPU blocks: 1280
INFO 03-22 20:15:07 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 3.48x
INFO 03-22 20:15:12 model_runner.py:1438] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  1.96it/s]

INFO 03-22 20:15:30 model_runner.py:1566] Graph capturing finished in 18 secs, took 0.24 GiB
INFO 03-22 20:15:30 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 23.69 seconds


In [5]:
SYSTEM_PROMPT = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)
t = CotTag.TASK.value
def get_openvla_prompt(instruction: str, task) -> str:
    return f"{SYSTEM_PROMPT} USER: What action should the robot take to {instruction.lower()}? ASSISTANT: {task}"
INSTRUCTION = "place the watermelon on the towel"
prompt = get_openvla_prompt(INSTRUCTION, t)
image = Image.open("../test.png")
print(prompt.replace(". ", ".\n"))
# print("Image size:", image.size)
dataset_statistics_path = os.path.join(path_to_converted_ckpt, "dataset_statistics.json")
if os.path.isfile(dataset_statistics_path):
    with open(dataset_statistics_path, "r") as f:
        norm_stats = json.load(f)
    vla.norm_stats = norm_stats

A chat between a curious user and an artificial intelligence assistant.
The assistant gives helpful, detailed, and polite answers to the user's questions.
USER: What action should the robot take to place the watermelon on the towel? ASSISTANT: TASK:


# VLLM

different prompt tests (not directly supported)

prepare the inputs 

In [6]:
async_prompts = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: What action should the robot take to place the watermelon on the towel? ASSISTANT: TASK: The task is to place the watermelon on the towel. The first step is to move the robotic arm towards the towel. PLAN: 1. Move to the right and forward. 2. Move down and grip the towel. 3. Move backward and up. 4. Move to the left. VISIBLE OBJECTS: the robot task [100, 1, 153, 105], the towel [160, 99, 220, 164], the towel [160, 99, 221, 165], table [20, 39, 239, 249], the robot task [100, 1, 154, 106] SUBTASK REASONING: The towel is to the right and slightly forward from the current robotic arm position. The robotic arm needs to move forward and up to reach the towel and grip it. SUBTASK: Move forward and up. MOVE REASONING: The robotic arm needs to move forward and up to reach the towel and grip it. MOVE: Move forward up. GRIPPER POSITION: [121, 91, 130, 87, 142, 87, 153, 88, 169, 95] ACTION: 塔瀬ܝĦ越ਿŸ"
# break async_prompts with CotTag keep value before the tag

prompts = []
for t in CotTag:
    # if t == CotTag.PLAN:
    #     break
    prompts.append(async_prompts.split(t.value)[0] + t.value)
    # print(prompts[-1]) 
from transformers.utils import TensorType
prompts_reason = prompts[:-1]
prompts_action = prompts[-1]

inputs_reason = [processor.tokenizer(p, return_tensors=TensorType.PYTORCH)['input_ids'].to(device) for p in prompts_reason]
inputs_action = [processor.tokenizer(prompts_action, return_tensors=TensorType.PYTORCH)['input_ids'].to(device)]
pixel_values = processor.image_processor(image, return_tensors=TensorType.PYTORCH)["pixel_values"].to(device, dtype=torch.bfloat16)

In [7]:
print(processor.tokenizer('VISIBLE OBJECTS:'))
print(processor.tokenizer.decode([29901]))

{'input_ids': [1, 478, 3235, 8979, 1307, 438, 29933, 17637, 29903, 29901], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
:


In [8]:
def get_outputs(inputs, pixel_values, sampling_params):
    for i in range(3):
        start = time.perf_counter()
        outputs = vla.vllm_inference(input_ids=inputs, pixel_values=pixel_values, sampling_params=sampling_params)
        print("Inference time:", time.perf_counter() - start)
        time.sleep(0.1)
        # return outputs


In [9]:
import threading
sampling_params = SamplingParams(temperature=0, max_tokens=60, stop_token_ids=[29901])
t1 = threading.Thread(target=get_outputs, args=(inputs_action, pixel_values, sampling_params), daemon=True)
t2 = threading.Thread(target=get_outputs, args=(inputs_reason, pixel_values, sampling_params), daemon=True)

t1.start()
time.sleep(0.2)
t2.start()


INFO 03-22 20:21:41 preprocess.py:238] Your model uses the legacy input pipeline instead of the new multi-modal processor. Please note that the legacy pipeline will be removed in a future release. For more details, see: https://github.com/vllm-project/vllm/issues/10114


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s, est. speed input: 2807.85 toks/s, output: 38.87 toks/s]


Inference time: 0.42378295701928437


Processed prompts:  62%|██████▎   | 5/8 [00:00<00:00,  7.30it/s, est. speed input: 3115.10 toks/s, output: 93.17 toks/s]Exception in thread Exception in thread Thread-8 (get_outputs):
Traceback (most recent call last):
  File "/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
Thread-7 (get_outputs):
Traceback (most recent call last):
  File "/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    self.run()
  File "/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_4179870/31885529

In [11]:
for o , p in zip(outputs, prompts):
    # print(p + ' ' + o.outputs[0].text)
    print(o.outputs[0].text)
    # print(o.outputs[0].token_ids)
    # generated_text = o.outputs[0].token_ids
    # print(generated_text)
    # print(len(generated_text))

NameError: name 'outputs' is not defined

In [ ]:
#profile memroy with torch
print(torch.cuda.memory_summary(device))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  18743 MiB |  19157 MiB | 112991 MiB |  94248 MiB |
|       from large pool |  18582 MiB |  18995 MiB | 109173 MiB |  90590 MiB |
|       from small pool |    160 MiB |    167 MiB |   3818 MiB |   3657 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  18743 MiB |  19157 MiB | 112991 MiB |  94248 MiB |
|       from large pool |  18582 MiB |  18995 MiB | 109173 MiB |